# 05 Review outputs

Review the latest inventory, rule-classification, and text-extraction outputs together before any rename planning or execution.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
EXPORT_REVIEW_SNAPSHOT = True


In [2]:
from src.reporting import detect_latest_outputs, load_optional_parquet, build_review_frame, review_summary
from src.inventory import ensure_inventory_schema

paths = detect_latest_outputs(OUTPUTS_DIR)
print(paths)

inv = load_optional_parquet(paths.inventory_path)
classified = load_optional_parquet(paths.classification_path)
text_df = load_optional_parquet(paths.text_path)

if inv is not None:
    inv = ensure_inventory_schema(inv)

review = build_review_frame(inv, classified, text_df)
print('Rows in review frame:', len(review))


ReviewPaths(inventory_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_smoke_test.parquet'), classification_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/rule_classification_smoke_test.parquet'), text_path=WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/inventory_with_text_smoke_test.parquet'))
Rows in review frame: 4


In [3]:
summary = review_summary(review)
pd.DataFrame([summary])


,rows,duplicates,junk_candidates,manual_review,text_errors,long_paths
0,4,2,1,2,1,0


In [4]:
display(review[['relative_path', 'suffix', 'size_bytes', 'rule_status', 'text_status', 'path_length']].head(20))
display(review['rule_status'].value_counts(dropna=False).rename_axis('rule_status').reset_index(name='count'))
display(review['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(review['suffix'].fillna('').value_counts(dropna=False).rename_axis('suffix').reset_index(name='count').head(20))


,relative_path,suffix,size_bytes,rule_status,text_status,path_length
0,docs/Thumbs.db,.db,4,archive_or_delete_candidate,unsupported,62
1,docs/a.txt,.txt,4,move_to_special_folder,ok,58
2,docs/PV15p473-01_PM_PER_environmental-approval...,.pdf,6,review,error,117
3,docs/b.txt,.txt,4,review,ok,58


,rule_status,count
0,review,2
1,archive_or_delete_candidate,1
2,move_to_special_folder,1


,text_status,count
0,ok,2
1,unsupported,1
2,error,1


,suffix,count
0,.txt,2
1,.db,1
2,.pdf,1


In [5]:
junk = review[review['rule_status'] == 'archive_or_delete_candidate'].copy()
duplicates = review[review.get('is_duplicate_hash', False).fillna(False)].copy()
manual_review = review[review['needs_manual_review']].copy()
text_errors = review[review['has_text_error']].copy()
long_paths = review[review['long_path_warning']].copy()
compliant = review[review['rule_status'] == 'compliant_keep_review_path'].copy()

display(junk[['relative_path', 'rule_reason', 'proposed_relative_target']].head(20))
display(duplicates[['relative_path', 'hash', 'duplicate_group_size', 'rule_status']].head(20))
display(text_errors[['relative_path', 'suffix', 'text_source', 'text_error']].head(20))
display(long_paths[['relative_path', 'path_length', 'filename_length', 'rule_status']].sort_values('path_length', ascending=False).head(20))
display(compliant[['relative_path', 'parsed_phase', 'parsed_doc_type', 'proposed_relative_target']].head(20))
display(manual_review[['relative_path', 'suffix', 'rule_reason', 'text_status', 'text_preview']].head(30))


,relative_path,rule_reason,proposed_relative_target
0,docs/Thumbs.db,junk filename from archive_rules,_DEPRECATED/Thumbs.db


,relative_path,hash,duplicate_group_size,rule_status
1,docs/a.txt,3899cbf2bbe606b305a65b559330d720,2,move_to_special_folder
3,docs/b.txt,3899cbf2bbe606b305a65b559330d720,2,review


,relative_path,suffix,text_source,text_error
2,docs/PV15p473-01_PM_PER_environmental-approval...,.pdf,pdf,PdfStreamError: Stream has ended unexpectedly


,relative_path,path_length,filename_length,rule_status


,relative_path,parsed_phase,parsed_doc_type,proposed_relative_target


,relative_path,suffix,rule_reason,text_status,text_preview
2,docs/PV15p473-01_PM_PER_environmental-approval...,.pdf,needs classification or rename mapping,error,
3,docs/b.txt,.txt,needs classification or rename mapping,ok,same


In [6]:
review_candidates = review[review['needs_manual_review'] | review['has_text_error'] | review['long_path_warning']].copy()
review_candidates = review_candidates.sort_values(['needs_manual_review', 'has_text_error', 'path_length', 'relative_path'], ascending=[False, False, False, True])
display(review_candidates[['relative_path', 'suffix', 'rule_status', 'rule_reason', 'text_status', 'path_length', 'text_preview']].head(50))


,relative_path,suffix,rule_status,rule_reason,text_status,path_length,text_preview
2,docs/PV15p473-01_PM_PER_environmental-approval...,.pdf,review,needs classification or rename mapping,error,117,
3,docs/b.txt,.txt,review,needs classification or rename mapping,ok,58,same


In [7]:
if EXPORT_REVIEW_SNAPSHOT:
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    review_path = OUTPUTS_DIR / 'review_snapshot_latest.parquet'
    review_csv = OUTPUTS_DIR / 'review_snapshot_latest.csv'
    review.to_parquet(review_path, index=False)
    review.to_csv(review_csv, index=False, encoding='utf-8-sig')
    print('Saved:', review_path)
    print('Saved:', review_csv)


Saved: c:\00_Developement\sch-file-organizer\data\outputs\review_snapshot_latest.parquet
Saved: c:\00_Developement\sch-file-organizer\data\outputs\review_snapshot_latest.csv
